## Import useful libraries

In [1]:
import os
import pickle

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType

## User settings

In [2]:
instrument = 'EUR/USD'
granularity = 'H1'
n_back = 200
lookahead = 4

column_y = 'pd_lead'    # 'spread_close_lead', 'volatility_lead'
percentiles = [100./3., 200. / 3.]

directory_data = 'output'

columns_x = ['volatility', 'return', 'diff_spread_close', 'diff_volume', 'day_sin', 'day_cos']

columns_non_time_series_to_keep = ['instrument', 'granularity', 'unix_epoch_s']
columns_non_time_series_to_keep.extend(columns_x)

## Initialize Spark session

In [3]:
conf = (
    SparkConf()
    .setAppName("MyApp")
    .set("spark.executor.memory", "100G")
    .set("spark.driver.memory", "100G")
    .set("spark.driver.maxResultSize", "100G")
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 17:00:57 WARN Utils: Your hostname, emily-MS-7B96, resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/07/05 17:00:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/05 17:00:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Initialize data source variables

In [4]:
training_set_filename = directory_data + '/df_training_and_testing_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'
training_set_filename_non_time_series = directory_data + '/df_training_and_testing_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'

## Load the data

In [5]:
df_time_series = spark.read.parquet(training_set_filename)
df_time_series.show(3)

26/07/05 17:01:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+-----------------------+--------------------+
|instrument|granularity|unix_epoch_s|             pd_lead|   spread_close_lead|     volatility_lead|             day_sin|             day_cos|            week_sin|            week_cos|          volatility|              return|   diff_spread_close|         diff_volume|    volatility_MA_12|        return_MA_12|diff_spread_close_MA_12|   diff_volume_MA_12|    volatility_MA_30|        return_MA_30|diff_spread_close_MA_30|   diff_volume_MA_30|    volatility_MA_50|       

In [6]:
df_non_time_series = (
    spark
    .read
    .parquet(training_set_filename_non_time_series)
    .select(*columns_non_time_series_to_keep)
    .orderBy('instrument', 'granularity', 'unix_epoch_s')
)
df_non_time_series.show(3)

+----------+-----------+------------+--------------------+--------------------+--------------------+-----------+------------+---------+
|instrument|granularity|unix_epoch_s|          volatility|              return|   diff_spread_close|diff_volume|     day_sin|  day_cos|
+----------+-----------+------------+--------------------+--------------------+--------------------+-----------+------------+---------+
|   EUR/USD|         H1|  1420502400|9.200000000000319E-4|5.949999999999012E-4|9.999999999843467E-6|        157|-6.37878E-12|      1.0|
|   EUR/USD|         H1|  1420506000|0.001019999999999...|-2.05000000000010...|1.000000000006551...|        -96|  0.25881904|0.9659258|
|   EUR/USD|         H1|  1420509600|0.001379999999999...|5.049999999999777E-4|                 0.0|         41|         0.5|0.8660254|
+----------+-----------+------------+--------------------+--------------------+--------------------+-----------+------------+---------+
only showing top 3 rows


## QA #1

In [7]:
df_time_series.select('instrument', 'granularity', 'unix_epoch_s', 'diff_volume').show(3)

+----------+-----------+------------+--------------------+
|instrument|granularity|unix_epoch_s|         diff_volume|
+----------+-----------+------------+--------------------+
|   EUR/USD|         H1|  1579770000|[-41, 165, -192, ...|
|   EUR/USD|         H1|  1579773600|[165, -192, 188, ...|
|   EUR/USD|         H1|  1579777200|[-192, 188, 257, ...|
+----------+-----------+------------+--------------------+
only showing top 3 rows


## Define a user-defined function to "stack" the lists into matrices

In [8]:
@F.udf(returnType=ArrayType(ArrayType(FloatType())))
def stack_time_series(*timeseries_list):
    result = []
    for ts in timeseries_list:
        result.append([float(x) for x in ts])
    return result

## Create matrices

In [9]:
column_objects = [F.col(c) for c in columns_x]

df_time_series = (
    df_time_series
    .withColumn('X', stack_time_series(*column_objects))
    .select('instrument', 'granularity', 'unix_epoch_s', column_y, 'X')
    .orderBy('instrument', 'granularity', 'unix_epoch_s')  
)

In [10]:
df_time_series.show(3)

[Stage 5:=====================================================>   (28 + 2) / 30]

+----------+-----------+------------+--------------------+--------------------+
|instrument|granularity|unix_epoch_s|             pd_lead|                   X|
+----------+-----------+------------+--------------------+--------------------+
|   EUR/USD|         H1|  1421391600| -0.3963000532984699|[[9.2E-4, 0.00102...|
|   EUR/USD|         H1|  1421395200|-0.20923384249771243|[[0.00102, 0.0013...|
|   EUR/USD|         H1|  1421398800| -0.3882418191902425|[[0.00138, 9.15E-...|
+----------+-----------+------------+--------------------+--------------------+
only showing top 3 rows


## Convert Spark dataframes to Pandas dataframes

In [11]:
pdf = (
    df_time_series
    .toPandas()
    .sort_values(by = ['instrument', 'granularity', 'unix_epoch_s'])
    .reset_index(drop = True)
)

In [12]:
pdf_non_time_series = (
    df_non_time_series
    .toPandas()
    .sort_values(by = ['instrument', 'granularity', 'unix_epoch_s'])
    .reset_index(drop = True)
)

## Define a class for encapsulating the training data

In [27]:
#
# Load useful libraries
#
import numpy as np

#
# Define a base class for encapsulating the training data
#
class Base():
    
    #
    # Constructor
    #
    def __init__(
        self,
        df,
        df_non_time_series,
        instrument,
        granularity,
        timestamp_column = 'unix_epoch_s',
        class_cutoff_percentiles = [100./3., 200. / 3.],
        column_y = 'pd_lead',
        columns_x = 'X',
        columns_x_components = ['volatility', 'return', 'diff_spread_close', 'diff_volume'],
        directory_output = 'output',
    ):
        self.instrument = instrument
        self.granularity = granularity
        self.timestamp_column = timestamp_column
        self.column_y = column_y
        self.columns_x = columns_x
        self.columns_x_components = columns_x_components
        self.directory_output = directory_output
        self.class_cutoff_percentiles = class_cutoff_percentiles

        self.df = (
            df[(df['instrument'] == self.instrument) & (df['granularity'] == self.granularity)]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )

        self.df_non_time_series = (
            df_non_time_series[
                (df_non_time_series['instrument'] == self.instrument) &
                (df_non_time_series['granularity'] == self.granularity)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )   

    #
    # Split data into training and test sets by timestamp ranges
    #
    def split_train_val_test_by_timestamps(
        self,
        min_timestamp_train,
        max_timestamp_train,
        min_timestamp_val,
        max_timestamp_val,
        min_timestamp_test,
        max_timestamp_test,
    ):
        df_train = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_train) & (self.df[self.timestamp_column] < max_timestamp_train)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )

        df_val = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_val) & (self.df[self.timestamp_column] < max_timestamp_val)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )
        
        df_test = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_test) & (self.df[self.timestamp_column] < max_timestamp_test)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )
        return df_train, df_val, df_test

    def outcome_inator(self, df, percentiles):
        y_array = df[self.column_y].values

        outcome = []
        for y in y_array:
            if y <= percentiles[0]:
                outcome.append([1, 0, 0])
            if percentiles[0] < y and y <= percentiles[1]:
                outcome.append([0, 1, 0])
            if percentiles[1] < y:
                outcome.append([0, 0, 1])

        return outcome
    
    #
    # Discretize the dependent variable
    #
    def define_outcome(self, df_train, df_val = None, df_test = None):
        y_array = df_train[self.column_y].values
        percentiles = np.percentile(y_array, self.class_cutoff_percentiles)

        df_train['outcome'] = self.outcome_inator(df_train, percentiles)
    
        if str(type(df_val)) != "<class 'NoneType'>":
            df_val['outcome'] = self.outcome_inator(df_val, percentiles)

        if str(type(df_test)) != "<class 'NoneType'>":
            df_test['outcome'] = self.outcome_inator(df_test, percentiles)
        
        return df_train, df_val, df_test

    #
    # Calculate timestamp difference across the data set
    #
    def get_timestamp_difference(self):
        min_timestamp = np.min(self.df[self.timestamp_column])
        max_timestamp = np.max(self.df[self.timestamp_column])
        diff = max_timestamp - min_timestamp
        return min_timestamp, max_timestamp, diff

    #
    # Train on a proportion of the data (earlier in time) and test on the rest (later in time)
    #
    # This is where ROC cutoffs are computed.
    #
    def train_and_test_by_proportion_and_timestamp(self, train_val_proportion = [0.7, 0.15]):
        min_timestamp, max_timestamp, diff = self.get_timestamp_difference()
        
        timestamp_start_train = min_timestamp
        timestamp_stop_train = min_timestamp + int(train_val_proportion[0] * diff)
        
        timestamp_start_val = timestamp_stop_train
        timestamp_stop_val = timestamp_stop_train + int(train_val_proportion[1] * diff)
        
        timestamp_start_test = timestamp_stop_val
        timestamp_stop_test = max_timestamp

        df_train, df_val, df_test = self.split_train_val_test_by_timestamps(
            timestamp_start_train, timestamp_stop_train,
            timestamp_start_val, timestamp_stop_val,
            timestamp_start_test, timestamp_stop_test,
        )
        
        df_train, df_val, df_test = self.define_outcome(df_train, df_val = df_val, df_test = df_test)

        X_train = np.array([np.array(i) for i in df_train['X'].to_numpy()])
        X_val = np.array([np.array(i) for i in df_val['X'].to_numpy()])
        X_test = np.array([np.array(i) for i in df_test['X'].to_numpy()])

        # normalize
        X_train_non_time_series = (
            self.df_non_time_series[self.df_non_time_series[self.timestamp_column] < timestamp_stop_train]
            [self.columns_x_components]
            .to_numpy()
        )
        the_mean_m_set = np.mean(X_train_non_time_series, axis = 0)
        the_std_m_set = np.std(X_train_non_time_series, axis = 0)
        m, n = X_train[0, :, :].shape
        the_mean = np.zeros([m, n])
        the_std = np.zeros([m, n])
        for i in range(0, m):
            the_mean[i, :] = the_mean_m_set[i]
            the_std[i, :] = the_std_m_set[i]
        X_train_norm = (X_train - the_mean) / the_std
        X_val_norm = (X_val - the_mean) / the_std
        X_test_norm = (X_test - the_mean) / the_std

        y_train = np.array([np.array(y) for y in df_train['outcome'].to_numpy()])
        y_val = np.array([np.array(y) for y in df_val['outcome'].to_numpy()])
        y_test = np.array([np.array(y) for y in df_test['outcome'].to_numpy()])
        
        self.to_lstm_trainer = {
            'train' : {
                'M' : X_train_norm,
                'y' : y_train,
            },
            'val' : {
                'M' : X_val_norm,
                'y' : y_val,
            },
            'test' : {
                'M' : X_test_norm,
                'y' : y_test,
            },
        }

    def save_data_for_LSTM(self):
        with open(self.directory_output + '/data.pickled', 'wb') as f:
            pickle.dump(self.to_lstm_trainer, f)

## Produce data for LSTM processing

In [29]:
b = Base(pdf, pdf_non_time_series, instrument, granularity, columns_x_components = columns_x)
b.train_and_test_by_proportion_and_timestamp()
b.save_data_for_LSTM()

## QA #2

In [30]:
with open('output/data.pickled', 'rb') as f:
    data = pickle.load(f)

In [31]:
data

{'train': {'M': array([[[-5.20012008e-01, -4.39474547e-01, -1.49539482e-01, ...,
           -4.07259488e-01,  4.54491710e-01,  7.72614835e-01],
          [ 5.01525509e-01, -1.73094971e-01,  4.25630716e-01, ...,
           -1.73094971e-01,  6.06934978e-01, -2.91153559e-01],
          [ 8.10690600e-02,  8.10690600e-02, -4.86628489e-06, ...,
           -2.43226645e-01,  2.43216912e-01, -1.62152719e-01],
          [ 8.31974297e-02, -5.10379907e-02,  2.16507548e-02, ...,
            1.39968932e-01,  3.59626892e-01,  5.87243475e-01],
          [-4.47884386e-04,  3.65573406e-01,  7.06650955e-01, ...,
            1.36556216e+00,  1.41374979e+00,  1.36556216e+00],
          [ 1.41448406e+00,  1.36629533e+00,  1.22501319e+00, ...,
            3.66283960e-01,  2.54389466e-04, -3.65775181e-01]],
  
         [[-4.39474547e-01, -1.49539482e-01, -5.24038914e-01, ...,
            4.54491710e-01,  7.72614835e-01,  1.47329120e+00],
          [-1.73094971e-01,  4.25630716e-01, -1.26714816e-01, ...,
     